# Pydantic is the new SQL

This notebook will illustrate the challenge of text-to-SQL approaches with LLMs,

> and why we believe **structured outputs with Pydantic** is the best way to query databases with LLMs.

### DSPy setup

This notebook does not use any particular features of DSPy -- it is just wrapping the structured output API from OpenAI.

In [28]:
import dspy
import os
lm = dspy.LM('openai/gpt-4o', api_key=os.getenv("OPENAI_API_KEY"))
dspy.configure(lm=lm)

In [89]:
lm("What is the future of database systems with AI?")

['The future of database systems with AI is poised to be transformative, as AI technologies continue to evolve and integrate more deeply into data management processes. Here are some key trends and potential developments:\n\n1. **Automated Database Management**: AI can automate many routine database management tasks, such as tuning, indexing, and query optimization. This reduces the need for manual intervention and allows database administrators to focus on more strategic tasks.\n\n2. **Intelligent Query Processing**: AI can enhance query processing by predicting query patterns and optimizing execution plans. This can lead to faster query responses and more efficient use of resources.\n\n3. **Enhanced Data Security**: AI can improve database security by detecting anomalies and potential threats in real-time. Machine learning algorithms can identify unusual access patterns and alert administrators to potential breaches.\n\n4. **Predictive Analytics**: AI-driven databases can provide adv

### Google BigQuery Setup

In [31]:
from google.cloud import bigquery

google_creds_file = "./my-google-creds.json"
project_id = ""
dataset_id = ""

import google.auth

from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file(google_creds_file)

bigquery_client = bigquery.Client(project=project_id, credentials=credentials)

### Google Data Marketplace

In [32]:
QUERY = (
    'SELECT name, number FROM `bigquery-public-data.usa_names.usa_1910_2013` '
    'WHERE state = "TX" '
    'LIMIT 100')
query_job = bigquery_client.query(QUERY)
rows = query_job.result()

for idx, row in enumerate(rows):
    if idx > 2:
        break
    print(row)

Row(('Mary', 895), {'name': 0, 'number': 1})
Row(('Roberta', 37), {'name': 0, 'number': 1})
Row(('Marguerite', 42), {'name': 0, 'number': 1})


# Execute BigQuery Query

In [93]:
def execute_query(query: str, client: bigquery.Client) -> bigquery.table.RowIterator:
    """
    Execute a BigQuery SQL query using the provided client
    
    Args:
        query: SQL query string to execute
        client: Authenticated BigQuery client
        
    Returns:
        Iterator of query results
    """
    query_job = client.query(query)
    return query_job.result()

### Database Schema Metadata Prompting

We prompt the LLM with the schema of the database it can query.

In [45]:
def get_table_schema(table_id: str) -> str:
    # Get the table details
    table = bigquery_client.get_table(table_id)
    
    # Build schema string
    schema_str = f"Schema for table {table_id}:\n"
    for schema_field in table.schema:
        schema_str += f"Name: {schema_field.name}, Type: {schema_field.field_type}, Mode: {schema_field.mode}\n"
    
    return schema_str

texas_names_schema = get_table_schema("bigquery-public-data.usa_names.usa_1910_2013")
print(texas_names_schema)

Schema for table bigquery-public-data.usa_names.usa_1910_2013:
Name: state, Type: STRING, Mode: NULLABLE
Name: gender, Type: STRING, Mode: NULLABLE
Name: year, Type: INTEGER, Mode: NULLABLE
Name: name, Type: STRING, Mode: NULLABLE
Name: number, Type: INTEGER, Mode: NULLABLE



### SQL as a Text Output

In [90]:
class SQLWriter(dspy.Signature):
    """Translate a natural language information need into a BigQuery query. In your rationale please be very clear about why you chose the specific query operators you did and why you do not need the operators you did not choose."""

    nl_command: str = dspy.InputField(desc="A natural language command with an underlying information need your db_query should answer.")
    db_schema: str = dspy.InputField(desc="The database schema you can query.")
    db_sql_query: str = dspy.OutputField()

In [92]:
bigquery_writer = dspy.ChainOfThought(SQLWriter)
texas_names_schema = get_table_schema("bigquery-public-data.usa_names.usa_1910_2013")

generated_query = bigquery_writer(
    nl_command = "What are the 5 most common names in Texas?",
    db_schema = texas_names_schema
)

db_sql_query = generated_query.db_sql_query

print(db_sql_query)

```sql
SELECT name, SUM(number) AS total_count
FROM `bigquery-public-data.usa_names.usa_1910_2013`
WHERE state = 'TX'
GROUP BY name
ORDER BY total_count DESC
LIMIT 5
```


In [94]:
execute_query(db_sql_query, bigquery_client)

BadRequest: 400 Syntax error: Unexpected identifier `` at [1:1]; reason: invalidQuery, location: query, message: Syntax error: Unexpected identifier `` at [1:1]

Location: US
Job ID: 5ca32c57-14ea-468c-9948-d8c163bec990


### Pydantic is the new SQL

In [71]:
from pydantic import BaseModel
from typing import Optional, List, Tuple
from google.cloud import bigquery

class BigQueryFilter(BaseModel):
    field: str
    operator: str 
    value: str | int | float

class BigQueryQuery(BaseModel):
    table: str
    fields: List[str]
    filters: Optional[List[BigQueryFilter]] = None
    limit: Optional[int] = None
    group_by: Optional[List[str]] = None
    sort_by: Optional[List[Tuple[str, str]]] = None  # List of (field, direction) tuples

    def to_sql(self) -> str:
        query = f"SELECT {', '.join(self.fields)} FROM `{self.table}`"
        
        if self.filters:
            conditions = []
            for f in self.filters:
                # Convert value to string with quotes if it's a string
                value = f'"{f.value}"' if isinstance(f.value, str) else str(f.value)
                conditions.append(f"{f.field} {f.operator} {value}")
            query += f" WHERE {' AND '.join(conditions)}"
            
        if self.group_by:
            query += f" GROUP BY {', '.join(self.group_by)}"
            
        if self.sort_by:
            sort_clauses = [f"{field} {direction}" for field, direction in self.sort_by]
            query += f" ORDER BY {', '.join(sort_clauses)}"
            
        if self.limit:
            query += f" LIMIT {self.limit}"
            
        return query

### DSPy `BigQueryWriter`

In [78]:
class BigQueryWriter(dspy.Signature):
    """Translate a natural language information need into a BigQuery query. In your rationale please be very clear about why you chose the specific query operators you did and why you do not need the operators you did not choose."""

    nl_command: str = dspy.InputField(desc="A natural language command with an underlying information need your db_query should answer.")
    db_schema: str = dspy.InputField(desc="The database schema you can query.")
    db_query: BigQueryQuery = dspy.OutputField()

# End-to-End Demo

In [83]:
bigquery_writer = dspy.ChainOfThought(BigQueryWriter)
texas_names_schema = get_table_schema("bigquery-public-data.usa_names.usa_1910_2013")

generated_query = bigquery_writer(
    nl_command = "What are the 5 most common names in Texas?",
    db_schema = texas_names_schema
)

print(generated_query)
db_query = generated_query.db_query

Prediction(
    reasoning='To find the 5 most common names in Texas, we need to focus on the `name` and `number` fields in the `usa_1910_2013` table. The `state` field will be used to filter the data to only include entries from Texas. We will sum the `number` of occurrences for each `name` to determine the total count of each name in Texas. After that, we will sort the names by their total count in descending order to identify the most common names. Finally, we will limit the results to the top 5 names. The `gender` and `year` fields are not relevant for this query as the command does not specify any gender or year constraints.',
    db_query=BigQueryQuery(table='bigquery-public-data.usa_names.usa_1910_2013', fields=['name', 'SUM(number) as total_count'], filters=[BigQueryFilter(field='state', operator='=', value='TX')], limit=5, group_by=['name'], sort_by=[('total_count', 'DESC')])
)


In [84]:
def pretty_print_query(query):
    """Pretty print a BigQueryQuery object"""
    print("BigQuery Query Details:")
    print(f"Table: {query.table}")
    print("\nFields:")
    for field in query.fields:
        print(f"- {field}")
    if query.filters:
        print("\nFilters:")
        for filter in query.filters:
            print(f"- {filter.field} {filter.operator} {filter.value}")
    if query.group_by:
        print("\nGroup By:")
        for group in query.group_by:
            print(f"- {group}")
    if query.limit:
        print(f"\nLimit: {query.limit}")

pretty_print_query(db_query)

BigQuery Query Details:
Table: bigquery-public-data.usa_names.usa_1910_2013

Fields:
- name
- SUM(number) as total_count

Filters:
- state = TX

Group By:
- name

Limit: 5


In [85]:
rows = execute_query(db_query.to_sql(), client=bigquery_client)

In [86]:
for row in rows:
    print(row)

Row(('James', 272793), {'name': 0, 'total_count': 1})
Row(('John', 235139), {'name': 0, 'total_count': 1})
Row(('Michael', 225320), {'name': 0, 'total_count': 1})
Row(('Robert', 220399), {'name': 0, 'total_count': 1})
Row(('David', 219028), {'name': 0, 'total_count': 1})
